# 08 - EDA con fuentes externas

**Objetivo.** Iniciar la lectura exploratoria sobre la base enriquecida en los notebooks 06 y 07. A diferencia del EDA inicial, esta etapa usa barrio/comuna oficiales, precios comparables en USD, distancias, conteos a 500 m y KPIs territoriales.

**Alcance de esta versión.** Se hacen cruces descriptivos y controles rápidos para dejar planteadas las hipótesis. Las pruebas estadísticas formales y la validación definitiva de hipótesis se desarrollan después sobre estas mismas salidas.

## 1. Preparación del entorno

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    import matplotlib.pyplot as plt
    HAS_SEABORN = False

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError('No se encontro la raiz del repo. Ejecutar dentro del proyecto.')


REPO_ROOT = find_repo_root()
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('REPORTS_DIR:', REPORTS_DIR)


## 2. Carga de base enriquecida y KPIs

In [ ]:
paths_requeridos = {
    'df_publicaciones_enriquecidas': PROCESSED_DIR / 'df_publicaciones_enriquecidas_v1.csv',
    'kpis_barrio': PROCESSED_DIR / 'kpis_barrio_v1.csv',
    'kpis_comuna': PROCESSED_DIR / 'kpis_comuna_v1.csv',
    'features_contextuales_barrio': PROCESSED_DIR / 'features_contextuales_barrio_v1.csv',
}

faltantes = [str(path.relative_to(REPO_ROOT)) for path in paths_requeridos.values() if not path.exists()]
if faltantes:
    raise FileNotFoundError('Faltan salidas de notebooks 06/07: ' + ', '.join(faltantes))

df = pd.read_csv(paths_requeridos['df_publicaciones_enriquecidas'], low_memory=False, dtype={'id_publicacion': str})
kpis_barrio = pd.read_csv(paths_requeridos['kpis_barrio'])
kpis_comuna = pd.read_csv(paths_requeridos['kpis_comuna'])
contexto_barrio = pd.read_csv(paths_requeridos['features_contextuales_barrio'])

print('Base enriquecida:', df.shape)
print('KPIs barrio:', kpis_barrio.shape, '| KPIs comuna:', kpis_comuna.shape)


## 3. Controles de cobertura territorial y variables nuevas

In [ ]:
cols_clave = [
    'barrio_oficial', 'comuna_oficial', 'lat', 'lon', 'precio_venta_usd', 'precio_mensual_usd',
    'precio_m2_usd', 'precio_por_ambiente_usd', 'dist_subte_m', 'dist_tren_m', 'dist_metrobus_m',
    'dist_espacio_verde_m', 'n_culturales_500m', 'n_gastronomia_500m', 'n_educacion_500m'
]
cols_clave = [c for c in cols_clave if c in df.columns]

cobertura = pd.DataFrame({
    'columna': cols_clave,
    'pct_no_nulo': [round(df[c].notna().mean() * 100, 2) for c in cols_clave],
    'n_no_nulo': [int(df[c].notna().sum()) for c in cols_clave],
    'n_unicos': [int(df[c].nunique(dropna=True)) for c in cols_clave],
}).sort_values('pct_no_nulo')

display(cobertura)
cobertura.to_csv(REPORTS_DIR / 'eda_fuentes_externas_cobertura_variables_v1.csv', index=False)

cobertura_fuente = (
    df.groupby('fuente_norm')[cols_clave]
    .apply(lambda x: x.notna().mean().mul(100).round(1))
    .reset_index()
)
display(cobertura_fuente)
cobertura_fuente.to_csv(REPORTS_DIR / 'eda_fuentes_externas_cobertura_por_fuente_v1.csv', index=False)


**Lectura.** La base enriquecida permite trabajar casi toda la oferta con barrio y comuna oficiales. Las variables de distancia dependen de tener coordenadas: Airbnb y las fuentes geocodificadas las tienen en distinta proporción, mientras que ZonaProp queda principalmente a nivel barrio declarado/oficial porque no informa dirección precisa.

## 4. Oferta y KPIs territoriales

In [ ]:
vista_comuna = [
    'comuna', 'n_alquiler', 'n_alquiler_temporal', 'n_venta', 'n_airbnb',
    'alquiler_por_1000_viviendas', 'airbnb_por_1000_viviendas',
    'alquiler_por_1000_hogares_inquilinos', 'publicaciones_por_1000_hab',
    'pct_viviendas_uso_temporal', 'pct_hogares_inquilinos', 'tamano_medio_hogar'
]
vista_comuna = [c for c in vista_comuna if c in kpis_comuna.columns]

tabla_comuna = kpis_comuna[vista_comuna].sort_values('airbnb_por_1000_viviendas', ascending=False)
display(tabla_comuna.round(2))
tabla_comuna.to_csv(PROCESSED_DIR / 'eda_comunas_censo_oferta_v1.csv', index=False)

vista_barrio = [
    'barrio_oficial', 'comuna', 'n_alquiler', 'n_alquiler_temporal', 'n_venta', 'n_airbnb',
    'airbnb_por_km2', 'mediana_precio_m2_venta_usd', 'mediana_precio_m2_alquiler_usd',
    'mediana_alquiler_mensual_usd', 'densidad_subte_km2', 'densidad_culturales_km2',
    'densidad_gastronomia_km2', 'pct_superficie_verde'
]
vista_barrio = [c for c in vista_barrio if c in kpis_barrio.columns]

tabla_barrio = kpis_barrio[vista_barrio].sort_values('airbnb_por_km2', ascending=False)
display(tabla_barrio.head(20).round(2))
tabla_barrio.to_csv(REPORTS_DIR / 'eda_fuentes_externas_kpis_barrio_top_airbnb_v1.csv', index=False)


## 5. Visualizaciones exploratorias

In [ ]:
if HAS_SEABORN:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    sns.scatterplot(
        data=kpis_comuna,
        x='pct_viviendas_uso_temporal',
        y='airbnb_por_1000_viviendas',
        size='n_airbnb',
        hue='comuna',
        palette='tab20',
        legend=False,
        ax=axes[0],
    )
    axes[0].set_title('Airbnb vs viviendas de uso temporal por comuna')
    axes[0].set_xlabel('% viviendas de uso temporal (Censo)')
    axes[0].set_ylabel('Airbnb por 1.000 viviendas')

    sns.scatterplot(
        data=kpis_barrio,
        x='densidad_gastronomia_km2',
        y='airbnb_por_km2',
        size='n_airbnb',
        hue='comuna',
        palette='tab20',
        legend=False,
        ax=axes[1],
    )
    axes[1].set_title('Airbnb y gastronomía turística por barrio')
    axes[1].set_xlabel('Densidad gastronomía / km²')
    axes[1].set_ylabel('Airbnb / km²')

    plt.tight_layout()
    plt.show()
else:
    print('Seaborn no disponible: se omiten visualizaciones.')


## 6. Señales preliminares para hipótesis

In [ ]:
lectura_preliminar_fuentes = pd.DataFrame([
    {
        'hipotesis/eje': 'H1 - atractivo turístico y alquiler temporario',
        'variables_disponibles': 'airbnb_por_km2, n_airbnb, densidad_gastronomia_km2, densidad_culturales_km2, pct_viviendas_uso_temporal',
        'lectura_inicial': 'Comparar concentración de Airbnb con gastronomía/cultura por barrio y viviendas temporales por comuna.',
        'siguiente_paso': 'Aplicar correlaciones Spearman y análisis visual controlando por comuna/centralidad.',
    },
    {
        'hipotesis/eje': 'H2 - servicios y perfil residencial',
        'variables_disponibles': 'n_educacion_500m, dist_hospitales_m, tamano_medio_hogar, ambientes_ref, precio_por_ambiente_usd',
        'lectura_inicial': 'Evaluar si zonas con más servicios y hogares más chicos muestran otra composición tipológica.',
        'siguiente_paso': 'Cruzar tipologías por comuna/barrio y realizar pruebas de asociación.',
    },
    {
        'hipotesis/eje': 'H3 - transporte y precio',
        'variables_disponibles': 'dist_subte_m, dist_tren_m, dist_metrobus_m, dist_transporte_masivo_m, precio_m2_usd',
        'lectura_inicial': 'Explorar si menor distancia a transporte se asocia con mayor precio/m².',
        'siguiente_paso': 'Usar Spearman/Kruskal y comparar bandas de distancia.',
    },
    {
        'hipotesis/eje': 'H4 - presión del alquiler temporario',
        'variables_disponibles': 'airbnb_por_1000_viviendas, alquiler_por_1000_viviendas, airbnb_por_cada_alquiler_permanente',
        'lectura_inicial': 'Identificar comunas/barrios con alta presión relativa del temporario frente al permanente.',
        'siguiente_paso': 'Construir rankings y contrastar con precios y disponibilidad residencial.',
    },
])

display(lectura_preliminar_fuentes)
lectura_preliminar_fuentes.to_csv(REPORTS_DIR / 'eda_fuentes_externas_lectura_preliminar_hipotesis_v1.csv', index=False)


## 7. Cierre

Este notebook deja planteada la lectura enriquecida: la base ya contiene precios comparables, ubicación oficial, variables de entorno y KPIs territoriales. La etapa siguiente debe convertir estas señales en validación formal de hipótesis, priorizando pocos contrastes bien elegidos y controlando las limitaciones de granularidad: publicación, barrio y comuna no responden a las mismas preguntas.